In [1]:
import pandas as pd
import numpy as np
import json

# loading sentiment scores
sentiment_df = pd.read_csv("../data/sentiment_scores.csv")
print(sentiment_df[['ticker', 'filing_id', 'remarks_sentiment']].to_string(index=False))

ticker            filing_id  remarks_sentiment
  AAPL 0000320193-25-000077          -0.043531
  AAPL 0000320193-25-000071          -0.037594
  AAPL 0000320193-25-000055          -0.049160
  AAPL 0000320193-26-000005          -0.032791
  MSFT 0000950170-25-010484          -0.035976
  MSFT 0001193125-25-256310          -0.002616
  MSFT 0001193125-26-027198          -0.017730
  MSFT 0000950170-25-100226          -0.021958
  MSFT 0000950170-25-061032          -0.023707
 GOOGL 0001652044-26-000012          -0.039610
 GOOGL 0001652044-25-000087          -0.038429
 GOOGL 0001652044-25-000056          -0.031499
  META 0001628280-26-003832          -0.048095
  META 0001326801-25-000050          -0.022372
  META 0001628280-25-036719          -0.004093
  META 0001628280-25-047114           0.016278
  AMZN 0001018724-26-000002           0.068406
  AMZN 0001018724-25-000121           0.070501
  AMZN 0001018724-25-000084           0.042228
  AMZN 0001018724-25-000034           0.021477


In [2]:
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

# loading EPS data for all tickers
TICKERS = ["AAPL", "MSFT", "GOOGL", "META", "AMZN"]
eps_dict = {}
for ticker in TICKERS:
    df = pd.read_csv(f"../data/{ticker}_earnings.csv")
    df['ticker'] = ticker
    eps_dict[ticker] = df

eps_df = pd.concat(eps_dict.values(), ignore_index=True)
print(eps_df.columns.tolist())

['Earnings Date', 'EPS Estimate', 'Reported EPS', 'Surprise(%)', 'ticker']


## step1: cleaning and aligning EPS data

In [3]:
# renaming columns
eps_df.columns = [c.strip() for c in eps_df.columns]
print("EPS columns:", eps_df.columns.tolist())
print(eps_df.head())

EPS columns: ['Earnings Date', 'EPS Estimate', 'Reported EPS', 'Surprise(%)', 'ticker']
               Earnings Date  EPS Estimate  Reported EPS  Surprise(%) ticker
0  2026-04-30 16:00:00-04:00          1.94           NaN          NaN   AAPL
1  2026-01-29 16:00:00-05:00          2.67          2.84         6.25   AAPL
2  2025-10-30 16:00:00-04:00          1.77          1.85         4.52   AAPL
3  2025-07-31 16:00:00-04:00          1.43          1.57         9.48   AAPL
4  2025-05-01 16:00:00-04:00          1.63          1.65         1.50   AAPL


## step2: computing earnings surprise

In [4]:
# finding the EPS estimate and actual columns, yfinance calls them 'EPS Estimate' and 'Reported EPS'
eps_df = eps_df.rename(columns={
    'EPS Estimate': 'eps_estimate',
    'Reported EPS': 'eps_actual'
})

# convert to numeric
eps_df['eps_estimate'] = pd.to_numeric(eps_df['eps_estimate'], errors='coerce')
eps_df['eps_actual'] = pd.to_numeric(eps_df['eps_actual'], errors='coerce')

# earnings surprise = (actual - estimate) / abs(estimate)
eps_df['earnings_surprise'] = (
    (eps_df['eps_actual'] - eps_df['eps_estimate']) / 
    eps_df['eps_estimate'].abs()
)

print("earnings surprise computed:")
print(eps_df[['ticker', 'eps_estimate', 'eps_actual', 'earnings_surprise']].dropna().head(10))

earnings surprise computed:
   ticker  eps_estimate  eps_actual  earnings_surprise
1    AAPL          2.67        2.84           0.063670
2    AAPL          1.77        1.85           0.045198
3    AAPL          1.43        1.57           0.097902
4    AAPL          1.63        1.65           0.012270
5    AAPL          2.35        2.40           0.021277
6    AAPL          1.60        1.64           0.025000
7    AAPL          1.34        1.40           0.044776
8    AAPL          1.51        1.53           0.013245
9    AAPL          2.10        2.18           0.038095
10   AAPL          1.39        1.46           0.050360


## step3: match filing IDs to dates

In [5]:
# so, we need to match each transcript filing to an earnings dat, and the filings ID contains the date - so we are going to extract it
import os
import re

sec_path = "../data/sec_filings/sec-edgar-filings"
filing_dates = []

for ticker in TICKERS:
    ticker_path = os.path.join(sec_path, ticker, "8-K")
    if not os.path.exists(ticker_path):
        continue
    for filing_id in os.listdir(ticker_path):
        filepath = os.path.join(ticker_path, filing_id, "full-submission.txt")
        if not os.path.exists(filepath):
            continue
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            text = f.read(2000)  # just read the header
        # Extract filed date from header
        match = re.search(r'FILED AS OF DATE:\s+(\d{8})', text)
        if match:
            date_str = match.group(1)
            date = pd.to_datetime(date_str, format='%Y%m%d')
            filing_dates.append({
                'ticker': ticker,
                'filing_id': filing_id,
                'filing_date': date
            })

dates_df = pd.DataFrame(filing_dates)
print(dates_df.head(10))

  ticker             filing_id filing_date
0   AAPL  0001140361-25-018400  2025-05-12
1   AAPL  0000320193-25-000077  2025-10-30
2   AAPL  0000320193-25-000071  2025-07-31
3   AAPL  0001140361-25-027340  2025-07-25
4   AAPL  0001140361-26-006577  2026-02-24
5   AAPL  0000320193-25-000055  2025-05-01
6   AAPL  0000320193-26-000005  2026-01-29
7   AAPL  0001140361-25-025275  2025-07-09
8   AAPL  0001140361-26-000199  2026-01-02
9   AAPL  0001140361-25-044561  2025-12-05


## step4: merge sentiment with dates

In [18]:
merged_df = sentiment_df.merge(dates_df, on=['ticker', 'filing_id'], how='left')
print(merged_df[['ticker', 'filing_id', 'filing_date', 'remarks_sentiment']].to_string(index=False))
print(merged_df.columns.tolist())

ticker            filing_id filing_date  remarks_sentiment
  AAPL 0000320193-25-000077  2025-10-30          -0.043531
  AAPL 0000320193-25-000071  2025-07-31          -0.037594
  AAPL 0000320193-25-000055  2025-05-01          -0.049160
  AAPL 0000320193-26-000005  2026-01-29          -0.032791
  MSFT 0000950170-25-010484  2025-01-29          -0.035976
  MSFT 0001193125-25-256310  2025-10-29          -0.002616
  MSFT 0001193125-26-027198  2026-01-28          -0.017730
  MSFT 0000950170-25-100226  2025-07-30          -0.021958
  MSFT 0000950170-25-061032  2025-04-30          -0.023707
 GOOGL 0001652044-26-000012  2026-02-04          -0.039610
 GOOGL 0001652044-25-000087  2025-10-29          -0.038429
 GOOGL 0001652044-25-000056  2025-07-23          -0.031499
  META 0001628280-26-003832  2026-01-28          -0.048095
  META 0001326801-25-000050  2025-04-30          -0.022372
  META 0001628280-25-036719  2025-07-30          -0.004093
  META 0001628280-25-047114  2025-10-29           0.0162

## step5: compute tone surprise

In [19]:
# sorting by ticker and date
merged_df = merged_df.sort_values(['ticker', 'filing_date']).reset_index(drop=True)

# computing tone surprise
tone_surprises = []

for idx, row in merged_df.iterrows():
    ticker = row['ticker']
    current_date = row['filing_date']
    current_sentiment = row['remarks_sentiment']
    
    # getting all previous filings for this ticker
    previous = merged_df[(merged_df['ticker'] == ticker) & (merged_df['filing_date'] < current_date)]['remarks_sentiment']
    
    if len(previous) == 0:
        tone_surprises.append(np.nan)
    else:
        tone_surprises.append(current_sentiment - previous.mean())

merged_df['tone_surprise'] = tone_surprises

print("tone surprise computed:")
print(merged_df[['ticker', 'filing_date', 'remarks_sentiment', 'tone_surprise']].to_string(index=False))

tone surprise computed:
ticker filing_date  remarks_sentiment  tone_surprise
  AAPL  2025-05-01          -0.049160            NaN
  AAPL  2025-07-31          -0.037594       0.011565
  AAPL  2025-10-30          -0.043531      -0.000154
  AAPL  2026-01-29          -0.032791       0.010637
  AMZN  2025-05-01           0.021477            NaN
  AMZN  2025-07-31           0.042228       0.020751
  AMZN  2025-10-30           0.070501       0.038648
  AMZN  2026-02-05           0.068406       0.023671
 GOOGL  2025-07-23          -0.031499            NaN
 GOOGL  2025-10-29          -0.038429      -0.006930
 GOOGL  2026-02-04          -0.039610      -0.004646
  META  2025-04-30          -0.022372            NaN
  META  2025-07-30          -0.004093       0.018279
  META  2025-10-29           0.016278       0.029510
  META  2026-01-28          -0.048095      -0.044699
  MSFT  2025-01-29          -0.035976            NaN
  MSFT  2025-04-30          -0.023707       0.012269
  MSFT  2025-07-30    

## step6: match earnings surprose to filing dates 

In [ ]:
# reloading eps data fresh from CSVs to avoid any corruption from previous cells
eps_frames = []
for ticker in TICKERS:
    df = pd.read_csv(f"../data/{ticker}_earnings.csv")
    df['ticker'] = ticker
    eps_frames.append(df)

eps_clean = pd.concat(eps_frames, ignore_index=True)

# the first column is the earnings date
eps_clean = eps_clean.rename(columns={eps_clean.columns[0]: 'earnings_date'})
eps_clean['earnings_date'] = pd.to_datetime(eps_clean['earnings_date'], utc=True, errors='coerce')
eps_clean['earnings_date'] = eps_clean['earnings_date'].dt.tz_localize(None)
eps_clean = eps_clean.rename(columns={
    'EPS Estimate': 'eps_estimate',
    'Reported EPS': 'eps_actual'
})
eps_clean['eps_estimate'] = pd.to_numeric(eps_clean['eps_estimate'], errors='coerce')
eps_clean['eps_actual'] = pd.to_numeric(eps_clean['eps_actual'], errors='coerce')
eps_clean['earnings_surprise'] = (
    (eps_clean['eps_actual'] - eps_clean['eps_estimate']) / eps_clean['eps_estimate'].abs()
)

print(eps_clean[['ticker', 'earnings_date', 'eps_estimate', 'eps_actual', 'earnings_surprise']].head(10).to_string(index=False))

ticker       earnings_date  eps_estimate  eps_actual  earnings_surprise
  AAPL 2026-04-30 20:00:00          1.94         NaN                NaN
  AAPL 2026-01-29 21:00:00          2.67        2.84           0.063670
  AAPL 2025-10-30 20:00:00          1.77        1.85           0.045198
  AAPL 2025-07-31 20:00:00          1.43        1.57           0.097902
  AAPL 2025-05-01 20:00:00          1.63        1.65           0.012270
  AAPL 2025-01-30 21:00:00          2.35        2.40           0.021277
  AAPL 2024-10-31 20:00:00          1.60        1.64           0.025000
  AAPL 2024-08-01 20:00:00          1.34        1.40           0.044776
  AAPL 2024-05-02 20:00:00          1.51        1.53           0.013245
  AAPL 2024-02-01 21:00:00          2.10        2.18           0.038095


In [24]:
eps_estimates = []
eps_actuals = []
earnings_surprises = []

for idx, row in merged_df.iterrows():
    ticker = row['ticker']
    filing_date = pd.to_datetime(row['filing_date'])
    
    ticker_eps = eps_clean[eps_clean['ticker'] == ticker].copy()
    
    if ticker_eps.empty or pd.isna(filing_date):
        eps_estimates.append(np.nan)
        eps_actuals.append(np.nan)
        earnings_surprises.append(np.nan)
        continue
    
    ticker_eps['date_diff'] = (ticker_eps['earnings_date'] - filing_date).abs()
    closest = ticker_eps.loc[ticker_eps['date_diff'].idxmin()]
    
    if closest['date_diff'].days <= 30:
        eps_estimates.append(closest['eps_estimate'])
        eps_actuals.append(closest['eps_actual'])
        earnings_surprises.append(closest['earnings_surprise'])
    else:
        eps_estimates.append(np.nan)
        eps_actuals.append(np.nan)
        earnings_surprises.append(np.nan)

merged_df['eps_estimate'] = eps_estimates
merged_df['eps_actual'] = eps_actuals
merged_df['earnings_surprise'] = earnings_surprises

print(merged_df[['ticker', 'filing_date', 'earnings_surprise', 'tone_surprise']].to_string(index=False))

ticker filing_date  earnings_surprise  tone_surprise
  AAPL  2025-05-01           0.012270            NaN
  AAPL  2025-07-31           0.097902       0.011565
  AAPL  2025-10-30           0.045198      -0.000154
  AAPL  2026-01-29           0.063670       0.010637
  AMZN  2025-05-01           0.169118            NaN
  AMZN  2025-07-31           0.272727       0.020751
  AMZN  2025-10-30           0.250000       0.038648
  AMZN  2026-02-05           0.000000       0.023671
 GOOGL  2025-07-23           0.050000            NaN
 GOOGL  2025-10-29           0.269912      -0.006930
 GOOGL  2026-02-04           0.068182      -0.004646
  META  2025-04-30           0.234165            NaN
  META  2025-07-30           0.210169       0.018279
  META  2025-10-29           0.086957       0.029510
  META  2026-01-28           0.085575      -0.044699
  MSFT  2025-01-29           0.035256            NaN
  MSFT  2025-04-30           0.074534       0.012269
  MSFT  2025-07-30           0.079882       0.

## step7: building the composite signal and saving

In [26]:
from scipy import stats

df_clean = merged_df.dropna(subset=['tone_surprise', 'earnings_surprise']).copy()

# normalizing both to z-scores so that they are on equal footing
df_clean['tone_surprise_z'] = stats.zscore(df_clean['tone_surprise'])
df_clean['earnings_surprise_z'] = stats.zscore(df_clean['earnings_surprise'])

# composite signal = average of both
df_clean['composite_signal'] = (df_clean['tone_surprise_z'] + df_clean['earnings_surprise_z']) / 2

print(f"rows with complete signal: {len(df_clean)} out of {len(merged_df)}")
print()
print(df_clean[['ticker', 'filing_date', 'tone_surprise_z', 'earnings_surprise_z', 'composite_signal']].to_string(index=False))

# saving
df_clean.to_csv("../data/features.csv", index=False)
print(f"\nSaved to data/features.csv!")

rows with complete signal: 15 out of 20

ticker filing_date  tone_surprise_z  earnings_surprise_z  composite_signal
  AAPL  2025-07-31         0.100720            -0.157575         -0.028428
  AAPL  2025-10-30        -0.514842            -0.754501         -0.634671
  AAPL  2026-01-29         0.051961            -0.545281         -0.246660
  AMZN  2025-07-31         0.583224             1.822482          1.202853
  AMZN  2025-10-30         1.523269             1.565075          1.544172
  AMZN  2026-02-05         0.736552            -1.266407         -0.264927
 GOOGL  2025-10-29        -0.870762             1.790591          0.459914
 GOOGL  2026-02-04        -0.750788            -0.494185         -0.622486
  META  2025-07-30         0.453358             1.113957          0.783658
  META  2025-10-29         1.043300            -0.281544          0.380878
  META  2026-01-28        -2.854638            -0.297196         -1.575917
  MSFT  2025-04-30         0.137679            -0.422239   